<a href="https://colab.research.google.com/github/kader-xai/data-science-roadmap/blob/main/module_26_sql_for_data_science.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> 🗄 **Module 26 — SQL for Data Science (the single biggest gap to fill)** &nbsp;·&nbsp; part of [`data-science-roadmap`](https://github.com/kader-xai/data-science-roadmap)


# Module 26 — SQL for Data Science

> A working data scientist spends roughly **40% of their day in SQL**. Every BI tool, every database, every modern data warehouse (BigQuery, Snowflake, Redshift, DuckDB) speaks SQL. If Python is the language for *modelling*, SQL is the language for *answering questions*.

This module uses **SQLite** (file-based, no server) so every cell runs anywhere — Colab, your laptop, anywhere. The syntax is 95% portable to Postgres / MySQL / BigQuery / Snowflake.

### What you'll cover
1. Why SQL — and how it fits with Pandas
2. Setup — build a 4-table demo database
3. **SELECT / FROM / WHERE** — the foundation
4. **Aggregations** — `COUNT`, `SUM`, `AVG`, `GROUP BY`, `HAVING`
5. **JOINs** — INNER, LEFT, FULL OUTER, self-join
6. **Subqueries vs CTEs** (`WITH ...`)
7. **Window functions** — `ROW_NUMBER`, `RANK`, `LAG`, running totals
8. **Date functions** + `CASE WHEN`
9. SQL ↔ Pandas — `pd.read_sql` and back
10. Practice with hidden answers


## 1 · Why SQL — and How It Fits with Pandas

| Task | Better in SQL | Better in Pandas |
|---|---|---|
| Pull data from a database | ✅ | ❌ |
| Aggregate billions of rows | ✅ | ❌ (won't fit in memory) |
| Quick exploration / charts | ❌ | ✅ |
| Reshape / pivot complex tables | ❌ | ✅ |
| Train ML models | ❌ | ✅ |

**Reality in industry:** SQL extracts and aggregates the data; Pandas takes it from there.

## 2 · Setup — Build a Demo Database

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(":memory:")     # in-memory database — no file written
cur = con.cursor()

cur.executescript("""
DROP TABLE IF EXISTS customers;
DROP TABLE IF EXISTS products;
DROP TABLE IF EXISTS orders;
DROP TABLE IF EXISTS order_items;

CREATE TABLE customers (
    customer_id INTEGER PRIMARY KEY,
    name TEXT, country TEXT, signup_date TEXT
);
CREATE TABLE products (
    product_id INTEGER PRIMARY KEY,
    name TEXT, category TEXT, price REAL
);
CREATE TABLE orders (
    order_id INTEGER PRIMARY KEY,
    customer_id INTEGER, order_date TEXT
);
CREATE TABLE order_items (
    order_id INTEGER, product_id INTEGER, qty INTEGER
);
""")
print("schema created")

In [ ]:
cur.executescript("""
INSERT INTO customers VALUES
  (1,'Ada',     'UK',     '2023-01-15'),
  (2,'Linus',   'Finland','2023-02-20'),
  (3,'Grace',   'USA',    '2023-03-10'),
  (4,'Guido',   'NL',     '2023-04-05'),
  (5,'Yukihiro','Japan',  '2024-01-12');

INSERT INTO products VALUES
  (10,'Laptop',     'Hardware', 1200.00),
  (11,'Keyboard',   'Hardware',   80.00),
  (12,'IDE Pro',    'Software',  149.00),
  (13,'Cloud Plan', 'Service',    49.00),
  (14,'GPU Hour',   'Service',     2.50);

INSERT INTO orders VALUES
  (100, 1, '2024-03-01'),
  (101, 2, '2024-03-05'),
  (102, 1, '2024-03-12'),
  (103, 3, '2024-04-02'),
  (104, 4, '2024-04-15'),
  (105, 2, '2024-05-01'),
  (106, 1, '2024-05-20'),
  (107, 5, '2024-06-03');

INSERT INTO order_items VALUES
  (100, 10, 1), (100, 11, 2),
  (101, 12, 1),
  (102, 13, 3),
  (103, 10, 1), (103, 14, 50),
  (104, 12, 2), (104, 13, 1),
  (105, 14, 100),
  (106, 11, 1),
  (107, 13, 6), (107, 14, 200);
""")
con.commit()

def q(sql):
    """Run SQL and return a Pandas DataFrame for nice display."""
    return pd.read_sql_query(sql, con)

q("SELECT name FROM sqlite_master WHERE type='table'")

## 3 · SELECT / FROM / WHERE — the Foundation

The shape of every query:

```sql
SELECT  <columns>          -- WHAT you want
FROM    <table>            -- WHERE the data lives
WHERE   <conditions>       -- WHICH rows
ORDER BY <column>          -- IN WHAT ORDER
LIMIT   <N>                -- HOW MANY
```

In [ ]:
q("SELECT * FROM customers")

In [ ]:
q("SELECT name, country FROM customers WHERE country = 'UK'")

In [ ]:
q("SELECT name, signup_date FROM customers ORDER BY signup_date DESC LIMIT 3")

### Comparison & logical operators (same as Python, slightly different syntax)

| SQL | Python |
|---|---|
| `=` | `==` |
| `<>` or `!=` | `!=` |
| `AND` / `OR` / `NOT` | `and` / `or` / `not` |
| `IN ('UK','USA')` | `country in {'UK','USA'}` |
| `BETWEEN 10 AND 20` | `10 <= x <= 20` |
| `LIKE 'A%'` | `s.startswith('A')` |
| `IS NULL` / `IS NOT NULL` | `is None` / `is not None` |

In [ ]:
q("SELECT name, country FROM customers WHERE country IN ('UK','USA') AND name LIKE 'A%'")

## 4 · Aggregations — `COUNT`, `SUM`, `AVG`, `GROUP BY`, `HAVING`

In [ ]:
q("""
SELECT
  COUNT(*)             AS n_customers,
  COUNT(DISTINCT country) AS n_countries
FROM customers
""")

In [ ]:
q("""
SELECT country, COUNT(*) AS n_customers
FROM customers
GROUP BY country
ORDER BY n_customers DESC
""")

**`HAVING` vs `WHERE`:** `WHERE` filters ROWS *before* aggregation, `HAVING` filters GROUPS *after*. You almost always want this distinction on interview questions.

In [ ]:
q("""
SELECT country, COUNT(*) AS n
FROM customers
GROUP BY country
HAVING COUNT(*) >= 1
ORDER BY n DESC
""")

## 5 · JOINs — Combining Tables

Picture two tables side by side, glued on a shared key:

```
customers           orders
+------+-----+      +------+----------+
| id=1 | Ada |  ←→  | 100  | cust_id=1|
| id=2 | Linus|     | 101  | cust_id=2|
+------+-----+      +------+----------+
```

| Join type | Keeps |
|---|---|
| `INNER JOIN` | only rows that match on BOTH sides |
| `LEFT JOIN` | all rows from LEFT table; NULLs where no match |
| `RIGHT JOIN` | all rows from RIGHT table |
| `FULL OUTER JOIN` | everything from both (SQLite doesn't support; emulate with UNION) |

In [ ]:
q("""
SELECT o.order_id, o.order_date, c.name AS customer
FROM orders o
INNER JOIN customers c ON c.customer_id = o.customer_id
ORDER BY o.order_date
""")

### Multi-table join — orders × customers × items × products

The most common real-world query: *"what did each customer actually buy?"*

In [ ]:
q("""
SELECT
  c.name      AS customer,
  o.order_id,
  p.name      AS product,
  oi.qty,
  p.price,
  oi.qty * p.price AS line_total
FROM orders o
JOIN customers   c  ON c.customer_id = o.customer_id
JOIN order_items oi ON oi.order_id   = o.order_id
JOIN products    p  ON p.product_id  = oi.product_id
ORDER BY o.order_date, o.order_id
""")

### LEFT JOIN — "who has zero orders?"

A customer with **no orders** would get dropped by an INNER JOIN. LEFT JOIN keeps them with NULLs.

In [ ]:
q("""
SELECT c.name, COUNT(o.order_id) AS n_orders
FROM customers c
LEFT JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY n_orders DESC
""")

## 6 · Subqueries vs CTEs (`WITH ...`)

Both let you build a query in steps. CTEs read top-to-bottom — much easier on the eyes.

In [ ]:
# Subquery — nested, hard to read past 2 levels
q("""
SELECT name, country FROM customers
WHERE customer_id IN (
    SELECT customer_id FROM orders
    GROUP BY customer_id HAVING COUNT(*) >= 2
)
""")

In [ ]:
# Same query as a CTE — reads top-to-bottom
q("""
WITH frequent_buyers AS (
    SELECT customer_id
    FROM orders
    GROUP BY customer_id
    HAVING COUNT(*) >= 2
)
SELECT c.name, c.country
FROM customers c
JOIN frequent_buyers f ON f.customer_id = c.customer_id
""")

**Industry rule of thumb:** when a query has more than one subquery or you have to break it into steps, use CTEs. They're easier to read, debug, and modify.

## 7 · Window Functions — the SQL Superpower

Window functions compute a value for each row **using a window of related rows** — without collapsing the table the way `GROUP BY` does. The syntax: `<func>() OVER (PARTITION BY ... ORDER BY ...)`.

| Function | What it does |
|---|---|
| `ROW_NUMBER()` | unique sequential number per row in window |
| `RANK()` / `DENSE_RANK()` | rank, with ties handled differently |
| `LAG(col, n)` / `LEAD(col, n)` | value from N rows back / forward |
| `SUM() OVER (...)` | running total |
| `AVG() OVER (... ROWS BETWEEN N PRECEDING AND CURRENT ROW)` | rolling mean |

In [ ]:
# Most recent order per customer
q("""
SELECT customer_id, order_id, order_date,
       ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY order_date DESC) AS rn
FROM orders
ORDER BY customer_id, rn
""")

In [ ]:
# Days since the customer's PREVIOUS order
q("""
SELECT
  customer_id, order_date,
  LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date) AS prev_date,
  julianday(order_date) - julianday(LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date)) AS days_since_prev
FROM orders
""")

In [ ]:
# Running revenue total per customer
q("""
WITH line AS (
    SELECT o.customer_id, o.order_date, oi.qty * p.price AS amount
    FROM orders o
    JOIN order_items oi ON oi.order_id = o.order_id
    JOIN products p ON p.product_id = oi.product_id
)
SELECT customer_id, order_date, amount,
       SUM(amount) OVER (PARTITION BY customer_id ORDER BY order_date) AS running_total
FROM line
ORDER BY customer_id, order_date
""")

## 8 · Date Functions + `CASE WHEN`

SQLite uses string dates with `julianday()` and `strftime()`. Postgres / BigQuery have richer functions but the patterns are the same.

In [ ]:
q("""
SELECT
  order_id,
  order_date,
  strftime('%Y',  order_date) AS year,
  strftime('%m',  order_date) AS month,
  strftime('%w',  order_date) AS weekday,
  julianday('now') - julianday(order_date) AS days_ago
FROM orders
LIMIT 5
""")

### `CASE WHEN` — SQL's if/elif/else

In [ ]:
q("""
SELECT
  customer_id,
  COUNT(*) AS n_orders,
  CASE
    WHEN COUNT(*) >= 3 THEN 'whale'
    WHEN COUNT(*) = 2  THEN 'repeat'
    WHEN COUNT(*) = 1  THEN 'one-time'
    ELSE 'never'
  END AS segment
FROM orders
GROUP BY customer_id
ORDER BY n_orders DESC
""")

## 9 · SQL ↔ Pandas

You'll constantly move between them. Two patterns to memorise:

In [ ]:
# 1) Pull a query result into a DataFrame
df = pd.read_sql_query("""
SELECT c.country, COUNT(*) AS n_orders
FROM orders o JOIN customers c ON c.customer_id = o.customer_id
GROUP BY c.country
""", con)
print(df)

import matplotlib.pyplot as plt
df.set_index("country")["n_orders"].plot.bar(title="Orders by country")
plt.show()

In [ ]:
# 2) Push a Pandas DataFrame INTO the database (e.g. for joining with native tables)
new = pd.DataFrame({"customer_id":[1,2,3], "is_premium":[1,0,1]})
new.to_sql("premium", con, if_exists="replace", index=False)

q("""
SELECT c.name, p.is_premium
FROM customers c
JOIN premium p ON p.customer_id = c.customer_id
""")

## 10 · Practice — Try Yourself

Use the database we built. Each exercise has a hidden answer below it.

**Ex 26.1** — Total revenue per CUSTOMER, sorted descending.
**Ex 26.2** — Top 3 PRODUCTS by total quantity sold.
**Ex 26.3** — For each customer, their LATEST order date and the number of days since (use `julianday('now')`).
**Ex 26.4** — Customers who have NEVER bought a 'Service' product. (Hint: LEFT JOIN + filter on NULL, or `NOT IN`.)
**Ex 26.5** — Monthly revenue trend — month, total revenue, # orders, running total.


In [ ]:
# Try yours first, then peek

# 26.1
print("\n--- 26.1 ---")
print(q("""
SELECT c.name, ROUND(SUM(oi.qty * p.price), 2) AS revenue
FROM customers c
JOIN orders o      ON o.customer_id = c.customer_id
JOIN order_items oi ON oi.order_id  = o.order_id
JOIN products p     ON p.product_id  = oi.product_id
GROUP BY c.name
ORDER BY revenue DESC
"""))

# 26.2
print("\n--- 26.2 ---")
print(q("""
SELECT p.name, SUM(oi.qty) AS units_sold
FROM products p
JOIN order_items oi ON oi.product_id = p.product_id
GROUP BY p.name
ORDER BY units_sold DESC
LIMIT 3
"""))

# 26.3
print("\n--- 26.3 ---")
print(q("""
SELECT c.name, MAX(o.order_date) AS last_order,
       julianday('now') - julianday(MAX(o.order_date)) AS days_ago
FROM customers c
JOIN orders o ON o.customer_id = c.customer_id
GROUP BY c.name
ORDER BY days_ago
"""))

# 26.4
print("\n--- 26.4 ---")
print(q("""
SELECT name FROM customers
WHERE customer_id NOT IN (
    SELECT DISTINCT o.customer_id
    FROM orders o
    JOIN order_items oi ON oi.order_id   = o.order_id
    JOIN products p     ON p.product_id   = oi.product_id
    WHERE p.category = 'Service'
)
"""))

# 26.5
print("\n--- 26.5 ---")
print(q("""
WITH monthly AS (
  SELECT strftime('%Y-%m', o.order_date) AS month,
         oi.qty * p.price AS amount,
         o.order_id
  FROM orders o
  JOIN order_items oi ON oi.order_id = o.order_id
  JOIN products p     ON p.product_id = oi.product_id
)
SELECT month,
       ROUND(SUM(amount), 2) AS revenue,
       COUNT(DISTINCT order_id) AS n_orders,
       ROUND(SUM(SUM(amount)) OVER (ORDER BY month), 2) AS running_total
FROM monthly
GROUP BY month
"""))

## 11 · Where This Scales

You now have ~95% of the SQL you'll write at most jobs. The remaining 5% is **dialect-specific** and learned on the job:

| Database | Things specific to it |
|---|---|
| **PostgreSQL** | `JSONB` operators, `array` type, `LATERAL` joins, full-text search |
| **BigQuery** | partitioned tables, `STRUCT` / `ARRAY` types, `APPROX_*` functions, ML inside SQL (`CREATE MODEL`) |
| **Snowflake** | semi-structured `VARIANT`, time travel, zero-copy clones |
| **DuckDB** | reads Parquet/Pandas natively, blazing fast on a single machine |
| **Spark SQL** | distributed; runs the same SQL across a cluster |

### Recommended next steps
- **dbt** — modular, version-controlled SQL pipelines (the standard for analytics engineering).
- **DuckDB** — drop-in replacement for SQLite that handles GBs locally, plays beautifully with Pandas.
- **Window functions practice** — `https://leetcode.com/problemset/database/` is the standard interview prep.

## Recap

✅ Read and write SELECT / WHERE / ORDER BY / LIMIT
✅ Aggregate with GROUP BY / HAVING and the right metric (`SUM` / `AVG` / `COUNT DISTINCT`)
✅ Pick the right JOIN flavour and read multi-table joins fluently
✅ Use CTEs (`WITH ...`) to keep complex queries readable
✅ Use window functions for rankings, lags, running totals — the SQL superpower
✅ Format dates and write `CASE WHEN` segments
✅ Move data fluently between SQL and Pandas via `pd.read_sql_query` and `df.to_sql`

### Next module
**Module 27 — Tree-Based Models & Gradient Boosting** — Random Forest, XGBoost, LightGBM, SHAP. The other workhorse alongside SQL.
